In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/metadata/stats.json
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/metadata/master.csv
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/metadata/class_map.json
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/metadata/val.csv
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/metadata/train.csv
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/metadata/test.csv
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/audio/val/machine_pump_normal/00000752.wav
/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/audio/val/machine_pump_normal/00000465.wav
/kaggle/input/datasets/asaavitupsoun

In [35]:
!pip install -q librosa audioread

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import librosa
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Device: cuda
GPU: Tesla T4


In [36]:
import os
for item in os.listdir('/kaggle/input'):
    print(item)

datasets


In [37]:
import os

base = '/kaggle/input/datasets'
for item in sorted(os.listdir(base)):
    print(item)

asaavitupsounder


In [38]:
import os
for item in os.listdir('/kaggle/input/datasets/asaavitupsounder'):
    print(item)

unified-domestic-audio-dataset


In [39]:
import os
for item in os.listdir('/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset'):
    print(item)

content


In [40]:
import os
for item in os.listdir('/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content'):
    print(item)

unified_dataset


In [41]:
!pip install -q librosa audioread

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import librosa
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Device: cuda
GPU: Tesla T4


In [42]:
SR         = 16000
N_MELS     = 128
N_FFT      = 1024
HOP        = 512
TARGET_LEN = 160000
TARGET_FRM = 313
DATA_ROOT  = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset'
META       = f'{DATA_ROOT}/metadata'
CSV        = f'{META}/master.csv'
CMAP       = f'{META}/class_map.json'

print('master.csv exists:   ', os.path.exists(CSV))
print('class_map.json exists:', os.path.exists(CMAP))

class AudioDataset(Dataset):
    def __init__(self, csv_path, class_map_path, split, augment=False):
        df           = pd.read_csv(csv_path, low_memory=False)
        self.df      = df[df['split'] == split].reset_index(drop=True)
        self.augment = augment

        # Fix paths from Colab → Kaggle
        self.df['output_path'] = self.df['output_path'].str.replace(
            '/content/unified_dataset',
            DATA_ROOT,
            regex=False
        )

        with open(class_map_path) as f:
            cm = json.load(f)
        self.label_to_idx = cm['label_to_idx']
        self.n_classes    = cm['n_classes']

    def __len__(self):
        return len(self.df)

    def _mel(self, path):
        try:
            y, _ = librosa.load(path, sr=SR, mono=True, duration=10.0)
        except Exception:
            return np.zeros((N_MELS, TARGET_FRM), dtype=np.float32)

        if len(y) < TARGET_LEN:
            y = np.pad(y, (0, TARGET_LEN - len(y)))
        else:
            y = y[:TARGET_LEN]

        if self.augment:
            shift = int(np.random.uniform(-0.1, 0.1) * len(y))
            y = np.roll(y, shift)
            gain = np.random.uniform(-3.0, 3.0)
            y = np.clip(y * (10 ** (gain / 20.0)), -1.0, 1.0)

        mel = librosa.feature.melspectrogram(
            y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
            n_mels=N_MELS, fmin=20, fmax=8000, power=2.0)
        log_mel = librosa.power_to_db(mel + 1e-9, ref=np.max).astype(np.float32)

        if log_mel.shape[1] < TARGET_FRM:
            log_mel = np.pad(log_mel, ((0,0),(0, TARGET_FRM - log_mel.shape[1])))
        else:
            log_mel = log_mel[:, :TARGET_FRM]

        if self.augment:
            f_start = np.random.randint(0, N_MELS - 20)
            log_mel[f_start:f_start + np.random.randint(0, 20), :] = log_mel.min()
            t_start = np.random.randint(0, TARGET_FRM - 40)
            log_mel[:, t_start:t_start + np.random.randint(0, 40)] = log_mel.min()

        log_mel = (log_mel - (-40.0)) / 20.0
        return log_mel

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel   = self._mel(str(row['output_path']))
        spec  = torch.from_numpy(mel).unsqueeze(0).repeat(3, 1, 1)
        label = self.label_to_idx.get(str(row['unified_label']), 0)
        return spec, torch.tensor(label, dtype=torch.long)

train_ds = AudioDataset(CSV, CMAP, 'train', augment=True)
val_ds   = AudioDataset(CSV, CMAP, 'val',   augment=False)
test_ds  = AudioDataset(CSV, CMAP, 'test',  augment=False)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,
                      num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                      num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                      num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'Classes: {train_ds.n_classes} | Device: {DEVICE}')

master.csv exists:    True
class_map.json exists: True
Train: 30,338 | Val: 6,488 | Test: 6,418
Classes: 40 | Device: cuda


In [43]:
def build_model(n_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, n_classes)
    )
    return model.to(DEVICE)

model = build_model(train_ds.n_classes)
total = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B0 | {total/1e6:.1f}M parameters')

EfficientNet-B0 | 4.1M parameters


In [45]:
import os, numpy as np, librosa
from pathlib import Path
from tqdm import tqdm
import pandas as pd, json

DATA_ROOT  = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset'
SPEC_DIR   = '/kaggle/working/spectrograms'
CSV        = f'{DATA_ROOT}/metadata/master.csv'

SR         = 16000
N_MELS     = 128
N_FFT      = 1024
HOP        = 512
TARGET_LEN = 160000
TARGET_FRM = 313

os.makedirs(SPEC_DIR, exist_ok=True)

df = pd.read_csv(CSV, low_memory=False)
df['output_path'] = df['output_path'].apply(
    lambda p: DATA_ROOT + '/audio/' + str(p).split('/audio/', 1)[1]
    if '/audio/' in str(p) else str(p)
)

def compute_and_save(row):
    src  = str(row['output_path'])
    # Mirror folder structure under SPEC_DIR
    rel  = src.split('/audio/', 1)[1]                        # e.g. train/speech/foo.wav
    out  = os.path.join(SPEC_DIR, rel).replace('.wav', '.npy')
    if os.path.exists(out):
        return True   # already done
    os.makedirs(os.path.dirname(out), exist_ok=True)
    if not os.path.exists(src):
        return False
    try:
        y, _ = librosa.load(src, sr=SR, mono=True, duration=10.0)
        if len(y) < TARGET_LEN:
            y = np.pad(y, (0, TARGET_LEN - len(y)))
        else:
            y = y[:TARGET_LEN]
        mel     = librosa.feature.melspectrogram(
            y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
            n_mels=N_MELS, fmin=20, fmax=8000, power=2.0)
        log_mel = librosa.power_to_db(mel + 1e-9, ref=np.max).astype(np.float32)
        if log_mel.shape[1] < TARGET_FRM:
            log_mel = np.pad(log_mel, ((0,0),(0, TARGET_FRM - log_mel.shape[1])))
        else:
            log_mel = log_mel[:, :TARGET_FRM]
        log_mel = (log_mel - (-40.0)) / 20.0
        np.save(out, log_mel)
        return True
    except Exception:
        return False

print(f'Pre-computing spectrograms for {len(df):,} clips...')
ok = 0
for _, row in tqdm(df.iterrows(), total=len(df), ncols=70):
    if compute_and_save(row):
        ok += 1

print(f'Done. {ok:,} / {len(df):,} spectrograms saved to {SPEC_DIR}')

Pre-computing spectrograms for 43,244 clips...


100%|██████████████████████████| 43244/43244 [06:43<00:00, 107.18it/s]

Done. 30,388 / 43,244 spectrograms saved to /kaggle/working/spectrograms


In [46]:
import os
import pandas as pd

DATA_ROOT = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset'
SPEC_DIR  = '/kaggle/working/spectrograms'
CSV       = f'{DATA_ROOT}/metadata/master.csv'

df = pd.read_csv(CSV, low_memory=False)

def npy_path(p):
    if '/audio/' in str(p):
        return os.path.join(SPEC_DIR, str(p).split('/audio/', 1)[1].replace('.wav', '.npy'))
    return ''

df['npy_path'] = df['output_path'].apply(npy_path)
df['npy_exists'] = df['npy_path'].apply(os.path.exists)

print(f'Spectrograms found   : {df["npy_exists"].sum():,}')
print(f'Spectrograms missing : {(~df["npy_exists"]).sum():,}')
print()
print('Missing by split:')
print(df[~df['npy_exists']]['split'].value_counts())
print()
print('Missing by source dataset:')
print(df[~df['npy_exists']]['source_dataset'].value_counts())
print()
print('Sample missing paths:')
print(df[~df['npy_exists']]['output_path'].head(3).tolist())

Spectrograms found   : 30,388
Spectrograms missing : 12,856

Missing by split:
split
train    9012
test     1927
val      1917
Name: count, dtype: int64

Missing by source dataset:
source_dataset
mimii    12856
Name: count, dtype: int64

Sample missing paths:
['/content/unified_dataset/audio/all/mimii/machine_fan_normal/00000117.wav', '/content/unified_dataset/audio/all/mimii/machine_fan_normal/00000565.wav', '/content/unified_dataset/audio/all/mimii/machine_fan_normal/00000305.wav']


In [48]:
import os

# Find where MIMII wav files actually are on Kaggle
mimii_base = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset/audio'
for item in sorted(os.listdir(mimii_base)):
    print(item)

test
train
val


In [49]:
import os, numpy as np, librosa
from tqdm import tqdm

DATA_ROOT = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset'
SPEC_DIR  = '/kaggle/working/spectrograms'
SR, N_MELS, N_FFT, HOP, TARGET_LEN, TARGET_FRM = 16000, 128, 1024, 512, 160000, 313

# Get only the missing MIMII rows
missing_df = df[~df['npy_exists']].copy()
print(f'Fixing {len(missing_df):,} missing MIMII spectrograms...')

def fix_mimii_path(p):
    # Replace /audio/all/mimii/{class}/ with /audio/{split}/{class}/
    # We don't know the split from the path so we try all three
    filename = os.path.basename(str(p))
    label    = str(p).split('/mimii/')[1].split('/')[0] if '/mimii/' in str(p) else ''
    if not label:
        return None
    for split in ['train', 'val', 'test']:
        candidate = os.path.join(DATA_ROOT, 'audio', split, label, filename)
        if os.path.exists(candidate):
            return candidate
    return None

ok = 0
for _, row in tqdm(missing_df.iterrows(), total=len(missing_df), ncols=70):
    src = fix_mimii_path(row['output_path'])
    if src is None:
        continue

    # Output npy path — use split/label/filename structure
    filename = os.path.basename(src)
    label    = str(row['output_path']).split('/mimii/')[1].split('/')[0]
    split    = row['split']
    out      = os.path.join(SPEC_DIR, split, label, filename.replace('.wav', '.npy'))
    os.makedirs(os.path.dirname(out), exist_ok=True)

    if os.path.exists(out):
        ok += 1
        continue

    try:
        y, _ = librosa.load(src, sr=SR, mono=True, duration=10.0)
        if len(y) < TARGET_LEN:
            y = np.pad(y, (0, TARGET_LEN - len(y)))
        else:
            y = y[:TARGET_LEN]
        mel     = librosa.feature.melspectrogram(
            y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
            n_mels=N_MELS, fmin=20, fmax=8000, power=2.0)
        log_mel = librosa.power_to_db(mel + 1e-9, ref=np.max).astype(np.float32)
        if log_mel.shape[1] < TARGET_FRM:
            log_mel = np.pad(log_mel, ((0,0),(0, TARGET_FRM - log_mel.shape[1])))
        else:
            log_mel = log_mel[:, :TARGET_FRM]
        log_mel = (log_mel - (-40.0)) / 20.0
        np.save(out, log_mel)
        ok += 1
    except Exception as e:
        pass

print(f'Done. {ok:,} / {len(missing_df):,} MIMII spectrograms saved.')

Fixing 12,856 missing MIMII spectrograms...


100%|██████████████████████████| 12856/12856 [00:12<00:00, 999.27it/s]

Done. 12,856 / 12,856 MIMII spectrograms saved.


In [50]:
import os, pandas as pd

DATA_ROOT = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset'
SPEC_DIR  = '/kaggle/working/spectrograms'

def npy_path_fixed(row):
    p     = str(row['output_path'])
    split = str(row['split'])
    if '/audio/all/mimii/' in p:
        label    = p.split('/mimii/')[1].split('/')[0]
        filename = os.path.basename(p).replace('.wav', '.npy')
        return os.path.join(SPEC_DIR, split, label, filename)
    elif '/audio/' in p:
        return os.path.join(SPEC_DIR, p.split('/audio/', 1)[1].replace('.wav', '.npy'))
    return ''

df2 = pd.read_csv(f'{DATA_ROOT}/metadata/master.csv', low_memory=False)
df2['npy_path']   = df2.apply(npy_path_fixed, axis=1)
df2['npy_exists'] = df2['npy_path'].apply(os.path.exists)

print(f'Spectrograms found  : {df2["npy_exists"].sum():,}')
print(f'Still missing       : {(~df2["npy_exists"]).sum():,}')

Spectrograms found  : 43,244
Still missing       : 0


In [51]:
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DATA_ROOT  = '/kaggle/input/datasets/asaavitupsounder/unified-domestic-audio-dataset/content/unified_dataset'
SPEC_DIR   = '/kaggle/working/spectrograms'
CSV        = f'{DATA_ROOT}/metadata/master.csv'
CMAP       = f'{DATA_ROOT}/metadata/class_map.json'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
N_MELS     = 128
TARGET_FRM = 313

def npy_path_fixed(row):
    p     = str(row['output_path'])
    split = str(row['split'])
    if '/audio/all/mimii/' in p:
        label    = p.split('/mimii/')[1].split('/')[0]
        filename = os.path.basename(p).replace('.wav', '.npy')
        return os.path.join(SPEC_DIR, split, label, filename)
    elif '/audio/' in p:
        return os.path.join(SPEC_DIR, p.split('/audio/', 1)[1].replace('.wav', '.npy'))
    return ''

class FastAudioDataset(Dataset):
    def __init__(self, csv_path, class_map_path, split, augment=False):
        df           = pd.read_csv(csv_path, low_memory=False)
        self.df      = df[df['split'] == split].reset_index(drop=True)
        self.augment = augment
        self.df['npy_path'] = self.df.apply(npy_path_fixed, axis=1)

        with open(class_map_path) as f:
            cm = json.load(f)
        self.label_to_idx = cm['label_to_idx']
        self.n_classes    = cm['n_classes']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        try:
            mel = np.load(str(row['npy_path'])).astype(np.float32)
        except Exception:
            mel = np.zeros((N_MELS, TARGET_FRM), dtype=np.float32)

        if self.augment:
            f_start = np.random.randint(0, N_MELS - 20)
            mel[f_start:f_start + np.random.randint(0, 20), :] = mel.min()
            t_start = np.random.randint(0, TARGET_FRM - 40)
            mel[:, t_start:t_start + np.random.randint(0, 40)] = mel.min()

        spec  = torch.from_numpy(mel).unsqueeze(0).repeat(3, 1, 1)
        label = self.label_to_idx.get(str(row['unified_label']), 0)
        return spec, torch.tensor(label, dtype=torch.long)

train_ds = FastAudioDataset(CSV, CMAP, 'train', augment=True)
val_ds   = FastAudioDataset(CSV, CMAP, 'val',   augment=False)
test_ds  = FastAudioDataset(CSV, CMAP, 'test',  augment=False)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,
                      num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                      num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                      num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'Classes: {train_ds.n_classes} | Device: {DEVICE}')

Train: 30,338 | Val: 6,488 | Test: 6,418
Classes: 40 | Device: cuda


In [52]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_model(n_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, n_classes)
    )
    return model.to(DEVICE)

model = build_model(train_ds.n_classes)
total = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B0 | {total/1e6:.1f}M parameters')

EfficientNet-B0 | 4.1M parameters


In [53]:
import time
from tqdm import tqdm

with open(CMAP) as f:
    cm = json.load(f)
label_to_idx = cm['label_to_idx']

class_counts = train_ds.df['unified_label'].value_counts()
weights = torch.ones(train_ds.n_classes)
for label, count in class_counts.items():
    idx = label_to_idx.get(label)
    if idx is not None:
        weights[idx] = 1.0 / (count + 1e-6)
weights = (weights / weights.sum() * train_ds.n_classes).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS    = 15
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr          = 1e-3,
    steps_per_epoch = len(train_dl),
    epochs          = EPOCHS,
    pct_start       = 0.1,
    anneal_strategy = 'cos',
)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for specs, labels in tqdm(loader, ncols=70, leave=False):
            specs, labels = specs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out  = model(specs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * labels.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

best_val_acc = 0.0
best_path    = '/kaggle/working/efficientnet_b0_best.pt'
history      = []
t_start      = time.time()

print(f'Training EfficientNet-B0 | {EPOCHS} epochs | {DEVICE}\n')
print(f'{"Ep":>3}  {"Tr Loss":>9}  {"Tr Acc":>8}  {"Vl Loss":>9}  {"Vl Acc":>8}  {"Time"}')
print('─' * 58)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    vl_loss, vl_acc = run_epoch(val_dl,   train=False)
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'train_loss': tr_loss,
                    'train_acc': tr_acc, 'val_loss': vl_loss, 'val_acc': vl_acc})

    star = ' ★' if vl_acc > best_val_acc else ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'val_acc': vl_acc,
            'n_classes': train_ds.n_classes,
            'class_map': label_to_idx,
        }, best_path)

    print(f'{epoch:>3}  {tr_loss:>9.4f}  {tr_acc:>7.2f}%  '
          f'{vl_loss:>9.4f}  {vl_acc:>7.2f}%  {elapsed:.0f}s{star}')

total_min = (time.time() - t_start) / 60
print(f'\nDone in {total_min:.1f} min | Best val acc: {best_val_acc:.2f}%')

Training EfficientNet-B0 | 15 epochs | cuda

 Ep    Tr Loss    Tr Acc    Vl Loss    Vl Acc  Time
──────────────────────────────────────────────────────────


  1     1.7808    67.43%     0.5332    85.56%  139s ★


  2     0.7221    85.34%     0.4841    83.48%  136s


  3     0.4861    91.12%     0.2682    94.00%  135s ★


  4     0.3301    93.53%     0.2187    95.72%  136s ★


...

 11     0.0305    98.74%     0.1780    97.63%  136s


 12     0.0295    99.09%     0.1427    97.81%  136s ★


 13     0.0180    99.20%     0.1423    98.23%  137s ★


 14     0.0171    99.36%     0.1477    98.18%  136s


 15     0.0125    99.45%     0.1430    98.17%  136s

Done in 34.1 min | Best val acc: 98.23%


In [54]:
from sklearn.metrics import classification_report
import numpy as np, torch
from tqdm import tqdm

ckpt = torch.load('/kaggle/working/efficientnet_b0_best.pt')
model.load_state_dict(ckpt['model_state'])
model.eval()

idx_to_label = {int(k): v for k, v in cm['idx_to_label'].items()}
label_names  = [idx_to_label[i] for i in range(train_ds.n_classes)]

all_preds, all_labels = [], []
with torch.no_grad():
    for specs, labels in tqdm(test_dl, ncols=70):
        preds = model(specs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f'Test accuracy (top-1): {test_acc:.2f}%\n')
print(classification_report(all_labels, all_preds,
      target_names=label_names, digits=3, zero_division=0))

100%|███████████████████████████████| 101/101 [00:07<00:00, 14.27it/s]

Test accuracy (top-1): 98.11%

                              precision    recall  f1-score   support

            activity_absence      0.983     0.984     0.983       999
            activity_cooking      0.996     0.985     0.991       274
             activity_eating      0.950     0.966     0.958       118
              activity_other      0.833     0.833     0.833       102
        activity_watching_tv      1.000     1.000     1.000       968
          alarm_bell_ringing      0.944     0.895     0.919        19
                       bells      0.964     1.000     0.981        53
                       birds      0.913     1.000     0.955        21
                     blender      0.667     0.929     0.776        28
                         cat      0.895     1.000     0.944        17
                 cat_outdoor      1.000     1.000     1.000        30
                chicken_coop      1.000     1.000     1.000        20
            cicadas_crickets      0.929     0.975     0.95